<a href="https://colab.research.google.com/github/Thanjaivalavan/M2-GenAI-AgenticAI/blob/main/02_naive_bayes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
Slides 24-30: Naive Bayes Spam Classifier
------------------------------------------
Implements, from scratch

    Bayes' Theorem:      P(C|X) = P(X|C) * P(C) / P(X)
    Naive independence:  P(x1,...,xn|C) ~= P(x1|C) * P(x2|C) * ... * P(xn|C)
    General equation:    P(C|x1,...,xn) proportional to  P(C) * PRODUCT_i P(xi|C)

    P(X) is dropped since it's identical for every class -> we just compare
    the un-normalized scores and pick the larger one.

Uses Laplace (add-1) smoothing so P(word|class)=0 never zeroes out a product.
"""

import re
import math
from collections import Counter


EMAILS = [
    ("You have WON a free prize! Claim your jackpot now", 1),
    ("Free entry to win a lottery prize, click now", 1),
    ("Congratulations winner, claim your free cash prize", 1),
    ("Limited offer, free gift, click the link now", 1),
    ("Hey, are we still meeting for lunch tomorrow?", 0),
    ("Please review the attached report before the meeting", 0),
    ("Can you send me the notes from class today?", 0),
    ("Let's catch up over coffee this weekend", 0),
]


def tokenize(text: str):
    return re.findall(r"[a-zA-Z]+", text.lower())


class NaiveBayesSpamClassifier:
    def __init__(self):
        self.vocab = set()
        self.class_word_counts = {0: Counter(), 1: Counter()}
        self.class_total_words = {0: 0, 1: 0}
        self.class_doc_counts = {0: 0, 1: 0}
        self.n_docs = 0

    def fit(self, texts, labels):
        self.n_docs = len(texts)
        for text, label in zip(texts, labels):
            tokens = tokenize(text)
            self.class_doc_counts[label] += 1
            self.class_total_words[label] += len(tokens)
            for tok in tokens:
                self.vocab.add(tok)
                self.class_word_counts[label][tok] += 1

    def prior(self, c):
        """P(C) = docs in class c / total docs"""
        return self.class_doc_counts[c] / self.n_docs

    def likelihood(self, word, c):
        """
        P(word | C), with Laplace (add-1) smoothing:
            (count(word, C) + 1) / (total_words(C) + |vocab|)
        """
        count = self.class_word_counts[c][word]
        return (count + 1) / (self.class_total_words[c] + len(self.vocab))

    def class_log_score(self, tokens, c):
        """
        log P(C) + sum_i log P(x_i | C)
        (log-space to avoid multiplying many tiny probabilities -> underflow;
        this is the same product from the slide, just numerically stable)
        """
        score = math.log(self.prior(c))
        for tok in tokens:
            if tok in self.vocab:
                score += math.log(self.likelihood(tok, c))
        return score

    def predict_proba(self, text):
        tokens = tokenize(text)
        scores = {c: self.class_log_score(tokens, c) for c in (0, 1)}

        # Convert back from log-space to a normalized probability for display.
        # (This just re-introduces P(X) as a normalizer across the 2 classes —
        # the classification decision itself never needed it, per the slide's
        # note that P(X) is dropped during comparison.)
        max_score = max(scores.values())
        exp_scores = {c: math.exp(s - max_score) for c, s in scores.items()}
        total = sum(exp_scores.values())
        return {c: v / total for c, v in exp_scores.items()}

    def predict(self, text):
        probs = self.predict_proba(text)
        return max(probs, key=probs.get), probs


def main():
    texts = [t for t, _ in EMAILS]
    labels = [lab for _, lab in EMAILS]

    model = NaiveBayesSpamClassifier()
    model.fit(texts, labels)

    print(f"Vocabulary size: {len(model.vocab)}")
    print(f"P(Spam) prior = {model.prior(1):.3f}   P(Ham) prior = {model.prior(0):.3f}\n")

    print("--- Predictions on training set ---")
    for text, label in zip(texts, labels):
        pred, probs = model.predict(text)
        tag = "SPAM" if pred == 1 else "HAM"
        print(f"[{tag:4s} P(spam)={probs[1]:.3f}] true={label}  \"{text[:50]}\"")

    new_text = "Congratulations! You are a lucky winner, claim your free prize"
    pred, probs = model.predict(new_text)
    print(f"\nNew message: \"{new_text}\"")
    print(f"P(Spam)={probs[1]:.3f}  P(Ham)={probs[0]:.3f} -> {'SPAM' if pred == 1 else 'HAM'}")


if __name__ == "__main__":
    main()


Vocabulary size: 51
P(Spam) prior = 0.500   P(Ham) prior = 0.500

--- Predictions on training set ---
[SPAM P(spam)=1.000] true=1  "You have WON a free prize! Claim your jackpot now"
[SPAM P(spam)=1.000] true=1  "Free entry to win a lottery prize, click now"
[SPAM P(spam)=0.999] true=1  "Congratulations winner, claim your free cash prize"
[SPAM P(spam)=0.998] true=1  "Limited offer, free gift, click the link now"
[HAM  P(spam)=0.002] true=0  "Hey, are we still meeting for lunch tomorrow?"
[HAM  P(spam)=0.002] true=0  "Please review the attached report before the meeti"
[HAM  P(spam)=0.003] true=0  "Can you send me the notes from class today?"
[HAM  P(spam)=0.004] true=0  "Let's catch up over coffee this weekend"

New message: "Congratulations! You are a lucky winner, claim your free prize"
P(Spam)=0.999  P(Ham)=0.001 -> SPAM
